# Grade Classifier - Google Colab

Ta beležnica trenira preprost PyTorch klasifikator, ki na podlagi odstotka napove oceno od 1 do 5. Podatki so umetno generirani.

Za hitrejše izvajanje izberi **Runtime > Change runtime type > T4 GPU**. Beležnica deluje tudi brez GPU-ja.

In [ ]:
!pip -q install numpy pandas matplotlib seaborn statsmodels torch

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Uporabljena naprava: {device}")

## Nastavitve eksperimenta

Po potrebi spremeni vrednosti v naslednji celici. `training_percent` in `eval_percent` skupaj ne smeta presegati 100.

In [ ]:
data_rows = 10_000
epoch_count = 1_000
training_percent = 80
eval_percent = 10
learning_rate = 0.05
random_seed = 42

if training_percent + eval_percent >= 100:
    raise ValueError("Odstotka za treniranje in validacijo morata skupaj biti manjša od 100.")

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

In [ ]:
def create_data(num_rows):
    percentages = np.round(np.random.random(num_rows) * 100, 1)
    grades = np.select(
        [percentages < 49, percentages < 62, percentages < 75, percentages < 89],
        [1, 2, 3, 4],
        default=5,
    )
    return pd.DataFrame({"percent": percentages, "grade": grades})

data = create_data(data_rows)
data.head()

In [ ]:
X = torch.tensor(data[["percent"]].values, dtype=torch.float32, device=device) / 100.0
y = torch.tensor(data["grade"].values - 1, dtype=torch.long, device=device)

indices = torch.randperm(len(data), device=device)
train_size = int(training_percent / 100 * len(data))
val_size = int(eval_percent / 100 * len(data))
train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

model = nn.Linear(1, 5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
validation_losses = []

for epoch in range(epoch_count):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X[train_idx]), y[train_idx])
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X[val_idx]), y[val_idx]).item()
        validation_losses.append((epoch + 1, val_loss))
        print(f"Epoha {epoch + 1:>5} | Validacijski loss: {val_loss:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(X[test_idx]).argmax(dim=1)
    accuracy = (predictions == y[test_idx]).float().mean().item() * 100

print(f"Končna natančnost: {accuracy:.2f}%")
print(f"Treniranje: {len(train_idx)} | validacija: {len(val_idx)} | testiranje: {len(test_idx)}")

if validation_losses:
    epochs, losses = zip(*validation_losses)
    plt.plot(epochs, losses)
    plt.xlabel("Epoha")
    plt.ylabel("Validacijski loss")
    plt.title("Napredek treniranja")
    plt.grid(True)
    plt.show()

## Preizkus napovedi

Spremeni `percent` in ponovno zaženi celico.

In [ ]:
percent = 75.0
if not 0 <= percent <= 100:
    raise ValueError("Odstotek mora biti med 0 in 100.")

with torch.no_grad():
    logits = model(torch.tensor([[percent / 100]], dtype=torch.float32, device=device))
    probabilities = torch.softmax(logits, dim=1).squeeze().cpu().numpy() * 100
    prediction = logits.argmax(dim=1).item() + 1

print(f"Napovedana ocena za {percent:.1f}%: {prediction}")
for grade, probability in enumerate(probabilities, start=1):
    print(f"Ocena {grade}: {probability:.1f}%")

In [ ]:
torch.save(model.state_dict(), "grade_classifier_colab.pth")
print("Model je shranjen kot grade_classifier_colab.pth")

from google.colab import files
files.download("grade_classifier_colab.pth")